# 41 — Competitor Response Engine

A closed-loop signal-to-action pipeline that monitors competitor signals, classifies response urgency (`ignore` / `watch` / `respond`), and only when urgency is `respond` fans out to generate a full counter-campaign brief, email copy, social posts, and a blog outline in parallel. Expensive generation is entirely conditional on signal strength — if the gate says ignore or watch, no further LLM calls are made.

## Framework Comparison — Conditional Generation Gate

The core pattern here is a **conditional generation gate**: classify urgency first, then only spin up downstream agents when the classification crosses the `respond` threshold.

| Framework | Equivalent construct |
|-----------|----------------------|
| **LangGraph** | Conditional edges — the `respond` edge activates the brief + fan-out subgraph; `ignore` and `watch` edges route directly to a terminal node, skipping all generation |
| **LangChain** | Router chains — `LLMRouterChain` selects the response branch based on the classification output, directing flow to either a no-op chain or the full brief + content chain |
| **CrewAI** | Conditional crews — a gate crew classifies urgency; a response crew is only instantiated and kicked off when urgency is `respond`, leaving the `ignore`/`watch` paths as early exits |

In [ ]:
!pip install openai pydantic python-dotenv

In [ ]:
import os

# Replace the value below with your real OpenAI API key before running
os.environ["OPENAI_API_KEY"] = "sk-..."

In [ ]:
"""Pydantic models for the competitor response engine."""

from typing import Literal

from pydantic import BaseModel, Field


class CompetitorSignal(BaseModel):
    source: Literal["news", "social", "press_release", "product_launch", "pricing"] = Field(
        description="Channel where the signal was detected."
    )
    headline: str = Field(description="Short headline or title of the signal.")
    summary: str = Field(description="2-4 sentence summary of what was detected.")
    competitor_name: str = Field(description="Name of the competitor who generated the signal.")
    brand_name: str = Field(description="Name of our brand being affected by this signal.")


class SignalBatch(BaseModel):
    brand_name: str = Field(description="Our brand name — used as context across all signals.")
    signals: list[CompetitorSignal] = Field(
        description="One or more competitor signals to evaluate together."
    )


ResponseUrgency = Literal["ignore", "watch", "respond"]


class SignalClassification(BaseModel):
    urgency: ResponseUrgency = Field(
        description=(
            "Urgency level: 'respond' for strong competitive threats or viral moments, "
            "'watch' for emerging patterns, 'ignore' for noise."
        )
    )
    reasoning: str = Field(
        description="Explanation of the urgency decision citing the strongest signal."
    )
    key_threat: str | None = Field(
        default=None,
        description="The primary competitive threat identified; populated when urgency is 'respond'.",
    )
    opportunity: str | None = Field(
        default=None,
        description="Counter-positioning opportunity for our brand; populated when urgency is 'respond'.",
    )


class CounterBrief(BaseModel):
    topic: str = Field(description="Campaign topic that directly addresses the competitive signal.")
    target_audience: str = Field(
        description="Audience segment most affected by the competitor move."
    )
    key_message: str = Field(
        description="Primary message that counter-positions our brand against the identified threat."
    )
    tone: str = Field(description="Desired tone for all content derived from this brief.")
    call_to_action: str = Field(
        description="The desired action the audience should take in response to this campaign."
    )
    counter_narrative: str = Field(
        description="The 1-2 sentence narrative that reframes the competitive threat in our favour."
    )


class EmailCopy(BaseModel):
    subject: str = Field(description="Email subject line (under 60 characters).")
    preview_text: str = Field(description="Email preview text (under 90 characters).")
    body: str = Field(description="Full email body in plain text (2-4 paragraphs).")
    cta_button_text: str = Field(description="Text for the primary call-to-action button.")


class SocialPost(BaseModel):
    platform: Literal["instagram", "linkedin", "twitter"] = Field(
        description="Target social platform."
    )
    caption: str = Field(description="Post caption optimised for the platform.")
    hashtags: list[str] = Field(description="3-5 relevant hashtags without the # symbol.")


class BlogOutline(BaseModel):
    title: str = Field(description="SEO-optimised blog post title.")
    meta_description: str = Field(
        description="Meta description for search engines (under 155 characters)."
    )
    sections: list[str] = Field(
        description="Ordered list of section headings for the blog post."
    )
    estimated_word_count: int = Field(
        description="Estimated total word count for the post."
    )


class ContentPack(BaseModel):
    topic: str = Field(description="Campaign topic.")
    email: EmailCopy = Field(description="Email marketing copy.")
    social: list[SocialPost] = Field(
        description="Social posts for Instagram, LinkedIn, and Twitter."
    )
    blog: BlogOutline = Field(description="Blog post outline.")


class ResponseEngineResult(BaseModel):
    brand_name: str = Field(description="Our brand name.")
    urgency: ResponseUrgency = Field(
        description="Classified urgency level for the signal batch."
    )
    classification: SignalClassification = Field(
        description="Full classification output from the gate agent."
    )
    counter_brief: CounterBrief | None = Field(
        default=None,
        description="Counter-campaign brief; only present when urgency is 'respond'.",
    )
    content_pack: ContentPack | None = Field(
        default=None,
        description="Full content pack; only present when urgency is 'respond'.",
    )

In [ ]:
"""System prompt constants for the competitor response engine."""

CLASSIFY_SYSTEM = """You are a competitive intelligence analyst and brand strategist.

Given a batch of competitor signals (news, social posts, press releases, product launches, or pricing changes), classify the overall response urgency for our brand.

Urgency levels:
- "respond": Strong competitive threat requiring immediate counter-action. Use this for competitor product launches that directly challenge our core offering, viral moments that shift audience perception, aggressive pricing moves that threaten retention, or press releases that reframe the market narrative against us.
- "watch": Emerging pattern worth monitoring but not yet requiring action. Use this for early signals with unclear impact, minor feature announcements, or isolated social mentions with low reach.
- "ignore": Noise with no meaningful competitive impact. Use this for tangential news, irrelevant announcements, or routine competitor activity that does not affect our positioning.

Rules:
- Evaluate all signals together as a batch — a cluster of weak signals can warrant "watch" even if each alone is ignorable
- Cite the strongest single signal in your reasoning field
- Only populate key_threat and opportunity when urgency is "respond"
- key_threat: the specific competitive threat our brand faces (1-2 sentences)
- opportunity: how this moment creates an opening for us to strengthen positioning (1-2 sentences)
- Return ONLY valid JSON matching the schema — no prose."""

BRIEF_SYSTEM = """You are a senior brand strategist and campaign planner.

Given a signal classification (key threat and opportunity) and our brand name, write a CounterBrief for a response campaign that directly addresses the competitive signal.

Rules:
- topic: frame around our strength, not the competitor's name — own the narrative
- target_audience: the segment most likely to be swayed by the competitor signal
- key_message: one clear sentence that counter-positions us against the identified threat
- tone: choose from professional / confident / empathetic / urgent based on the nature of the threat
- call_to_action: specific, measurable action the audience should take (include a destination or method)
- counter_narrative: 1-2 sentences that reframe the competitive threat as a reason to choose us instead
- Do NOT mention the competitor by name in the brief — focus entirely on our brand's value
- Return ONLY valid JSON matching the schema — no prose."""

EMAIL_SYSTEM = """You are an expert email marketing copywriter.

Given a counter-campaign brief (topic, target audience, key message, tone, call to action, counter narrative), write compelling email marketing copy that responds to a competitive threat.

Rules:
- subject: punchy, under 60 characters, creates curiosity or urgency without mentioning the competitor
- preview_text: extends the subject hook, under 90 characters
- body: 2-4 paragraphs; lead with the audience's concern or aspiration, introduce our differentiated offer, build conviction, close with the CTA
- cta_button_text: 2-5 words, action-oriented
- Weave the counter_narrative naturally into the body — do not paste it verbatim
- Match the specified tone exactly
- Return ONLY valid JSON matching the schema — no prose."""

SOCIAL_SYSTEM = """You are an expert social media content strategist.

Given a counter-campaign brief, create three social media posts — one each for Instagram, LinkedIn, and Twitter — that reinforce our brand's positioning in response to a competitive moment.

Rules:
- Instagram: visual-first language, emotive, 150-220 characters, 3-5 hashtags
- LinkedIn: professional, insight-led, 200-300 characters, 3-4 hashtags
- Twitter: concise, punchy, under 280 characters including hashtags, 2-3 hashtags
- Each caption must feel native to its platform — do not copy-paste
- Do NOT mention the competitor by name
- Match the specified tone exactly
- Return ONLY valid JSON: an object with a "posts" array of three SocialPost objects — no prose."""

BLOG_SYSTEM = """You are an expert SEO content strategist.

Given a counter-campaign brief, produce a blog post outline that establishes our brand's authority and addresses the audience concern surfaced by the competitive signal.

Rules:
- title: include the primary keyword naturally, under 65 characters, angle toward the reader's concern
- meta_description: summarise the post value proposition, under 155 characters, include the keyword
- sections: 5-8 section headings — start with an intro that acknowledges the market shift, include a problem/context section, 2-4 substantive sections on our approach or differentiation, and a conclusion/CTA
- estimated_word_count: realistic total for the outlined sections (typically 800-1500)
- Do NOT mention the competitor by name in any section heading
- Return ONLY valid JSON matching the schema — no prose."""

In [ ]:
"""Competitor response engine workflow.

Signal-to-action pipeline:
1. Gate agent classifies urgency (ignore / watch / respond).
2. Only on "respond": build a counter-campaign brief.
3. Only on "respond": fan out email, social, and blog generation in parallel.
"""

import json
import os
from concurrent.futures import ThreadPoolExecutor, as_completed

from openai import OpenAI

_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
_MODEL = "gpt-4o-mini"


def _classify(batch: SignalBatch) -> SignalClassification:
    """Gate agent: classify urgency for the entire signal batch."""
    user_content = json.dumps(batch.model_dump())
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": CLASSIFY_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "SignalClassification",
                "strict": True,
                "schema": SignalClassification.model_json_schema(),
            },
        },
    )
    return SignalClassification.model_validate_json(resp.choices[0].message.content)


def _build_brief(brand_name: str, classification: SignalClassification) -> CounterBrief:
    """Build a counter-campaign brief from the gate classification."""
    user_content = json.dumps(
        {
            "brand_name": brand_name,
            "key_threat": classification.key_threat,
            "opportunity": classification.opportunity,
            "reasoning": classification.reasoning,
        }
    )
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": BRIEF_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "CounterBrief",
                "strict": True,
                "schema": CounterBrief.model_json_schema(),
            },
        },
    )
    return CounterBrief.model_validate_json(resp.choices[0].message.content)


def _write_email(brief: CounterBrief) -> EmailCopy:
    """Generate email copy from the counter-campaign brief."""
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": EMAIL_SYSTEM},
            {"role": "user", "content": json.dumps(brief.model_dump())},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "EmailCopy",
                "strict": True,
                "schema": EmailCopy.model_json_schema(),
            },
        },
    )
    return EmailCopy.model_validate_json(resp.choices[0].message.content)


def _write_social(brief: CounterBrief) -> list[SocialPost]:
    """Generate platform-native social posts from the counter-campaign brief."""
    social_schema = {
        "type": "object",
        "properties": {
            "posts": {
                "type": "array",
                "items": SocialPost.model_json_schema(),
            }
        },
        "required": ["posts"],
        "additionalProperties": False,
    }
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": SOCIAL_SYSTEM},
            {"role": "user", "content": json.dumps(brief.model_dump())},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "SocialPosts",
                "strict": True,
                "schema": social_schema,
            },
        },
    )
    data = json.loads(resp.choices[0].message.content)
    return [SocialPost.model_validate(p) for p in data["posts"]]


def _write_blog(brief: CounterBrief) -> BlogOutline:
    """Generate a blog outline from the counter-campaign brief."""
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": BLOG_SYSTEM},
            {"role": "user", "content": json.dumps(brief.model_dump())},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "BlogOutline",
                "strict": True,
                "schema": BlogOutline.model_json_schema(),
            },
        },
    )
    return BlogOutline.model_validate_json(resp.choices[0].message.content)


def run(batch: SignalBatch) -> ResponseEngineResult:
    """Run the full competitor response pipeline.

    Steps:
    1. Gate agent classifies signal urgency.
    2. If not "respond", return early with no brief or content.
    3. Build counter-campaign brief from the classification.
    4. Fan out email / social / blog generation in parallel.
    5. Return full ResponseEngineResult.
    """
    classification = _classify(batch)

    if classification.urgency != "respond":
        return ResponseEngineResult(
            brand_name=batch.brand_name,
            urgency=classification.urgency,
            classification=classification,
        )

    brief = _build_brief(batch.brand_name, classification)

    tasks = {
        "email": lambda: _write_email(brief),
        "social": lambda: _write_social(brief),
        "blog": lambda: _write_blog(brief),
    }

    results: dict = {}
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {executor.submit(fn): key for key, fn in tasks.items()}
        for future in as_completed(futures):
            results[futures[future]] = future.result()

    content_pack = ContentPack(
        topic=brief.topic,
        email=results["email"],
        social=results["social"],
        blog=results["blog"],
    )

    return ResponseEngineResult(
        brand_name=batch.brand_name,
        urgency=classification.urgency,
        classification=classification,
        counter_brief=brief,
        content_pack=content_pack,
    )

In [ ]:
# ---------------------------------------------------------------------------
# Scenario A — product launch that directly threatens our core offering
# Expected: urgency = "respond" → counter brief + full content pack generated
# ---------------------------------------------------------------------------
SCENARIO_A = SignalBatch(
    brand_name="Flowdesk",
    signals=[
        CompetitorSignal(
            source="product_launch",
            headline="NovaCRM launches free tier with full automation suite",
            summary=(
                "NovaCRM announced a permanent free tier that includes their full marketing "
                "automation suite — email sequences, CRM, and landing pages. The launch is "
                "backed by a $10M Series A and is explicitly targeting small businesses "
                "currently using Flowdesk. Early adopters report the product is feature-complete "
                "for most SMB use cases."
            ),
            competitor_name="NovaCRM",
            brand_name="Flowdesk",
        ),
        CompetitorSignal(
            source="social",
            headline="NovaCRM free tier announcement goes viral on LinkedIn",
            summary=(
                "NovaCRM's LinkedIn post announcing the free tier has over 4,200 reactions and "
                "600 comments in under 12 hours. Multiple small business owners are tagging "
                "Flowdesk and asking whether we plan to respond. Several threads compare "
                "pricing directly and call out Flowdesk's lack of a free plan."
            ),
            competitor_name="NovaCRM",
            brand_name="Flowdesk",
        ),
    ],
)


def _print_result(label: str, result) -> None:
    print(f"\n{'=' * 60}")
    print(f"  {label}")
    print(f"{'=' * 60}")
    print(f"Brand:   {result.brand_name}")
    print(f"Urgency: {result.urgency.upper()}")
    print(f"Reason:  {result.classification.reasoning}")

    if result.urgency == "respond":
        print(f"\nKey Threat:  {result.classification.key_threat}")
        print(f"Opportunity: {result.classification.opportunity}")

        brief = result.counter_brief
        print("\n--- Counter Brief ---")
        print(f"Topic:     {brief.topic}")
        print(f"Audience:  {brief.target_audience}")
        print(f"Message:   {brief.key_message}")
        print(f"Tone:      {brief.tone}")
        print(f"CTA:       {brief.call_to_action}")
        print(f"Narrative: {brief.counter_narrative}")

        pack = result.content_pack
        print("\n--- Email ---")
        print(f"Subject:  {pack.email.subject}")
        print(f"Preview:  {pack.email.preview_text}")
        print(f"CTA Btn:  {pack.email.cta_button_text}")
        print(f"\n{pack.email.body}")

        print("\n--- Social Posts ---")
        for post in pack.social:
            print(f"\n[{post.platform.upper()}]")
            print(post.caption)
            print("  #" + "  #".join(post.hashtags))

        print("\n--- Blog Outline ---")
        print(f"Title: {pack.blog.title}")
        print(f"Meta:  {pack.blog.meta_description}")
        print(f"~{pack.blog.estimated_word_count} words")
        for i, section in enumerate(pack.blog.sections, 1):
            print(f"  {i}. {section}")
    else:
        print("\n[No content generated — urgency below response threshold]")


print("Running Scenario A: Competitor product launch (expect: respond)")
result_a = run(SCENARIO_A)
_print_result("SCENARIO A — Product Launch Threat", result_a)

In [ ]:
# ---------------------------------------------------------------------------
# Scenario B — minor competitor publishes a tangential blog post
# Expected: urgency = "ignore" → no brief or content pack generated
# ---------------------------------------------------------------------------
SCENARIO_B = SignalBatch(
    brand_name="Flowdesk",
    signals=[
        CompetitorSignal(
            source="news",
            headline="SmallReach publishes guide to cold email best practices",
            summary=(
                "SmallReach, a niche cold-email tool with under 500 customers, published a "
                "1,500-word blog post on cold email deliverability tips. The post has no paid "
                "distribution, targets enterprise sales teams (not our SMB segment), and does "
                "not mention Flowdesk or any competing product."
            ),
            competitor_name="SmallReach",
            brand_name="Flowdesk",
        ),
    ],
)

print("Running Scenario B: Minor blog post (expect: ignore)")
result_b = run(SCENARIO_B)
_print_result("SCENARIO B — Minor Blog Post", result_b)

## Starter Exercise

Add a **watch** path: when urgency is `watch`, generate a monitoring brief (competitor summary + suggested tracking keywords) without producing any campaign content. Where in the workflow would you add this branch and what new schema model would you need?

Think about:
- What fields belong in a `MonitoringBrief` model? (A short summary of the emerging threat and a list of keywords to track.)
- In `run()`, where would you insert the `elif urgency == "watch"` branch relative to the existing `if classification.urgency != "respond"` guard?
- Should the `ResponseEngineResult` model gain a new optional `monitoring_brief` field, or would you return a separate result type?

In [ ]:
# ---------------------------------------------------------------------------
# Answer key — watch path: MonitoringBrief schema + workflow branch
# ---------------------------------------------------------------------------

# 1. New schema model
class MonitoringBrief(BaseModel):
    competitor_summary: str = Field(
        description="1-2 sentence summary of the emerging competitive pattern being watched."
    )
    tracking_keywords: list[str] = Field(
        description="5-10 keywords or phrases to monitor across news, social, and search."
    )


# 2. System prompt for the watch path
WATCH_SYSTEM = """You are a competitive intelligence analyst.

Given a signal classification marked as 'watch', produce a brief monitoring summary.

Rules:
- competitor_summary: 1-2 sentences describing the emerging pattern and why it warrants watching
- tracking_keywords: 5-10 specific terms to monitor (brand names, feature names, market terms)
- Return ONLY valid JSON matching the schema — no prose."""


# 3. New helper function
def _build_monitoring_brief(
    brand_name: str, classification: SignalClassification
) -> MonitoringBrief:
    """Generate a lightweight monitoring brief for watch-level signals."""
    user_content = json.dumps(
        {
            "brand_name": brand_name,
            "reasoning": classification.reasoning,
        }
    )
    resp = _client.chat.completions.create(
        model=_MODEL,
        messages=[
            {"role": "system", "content": WATCH_SYSTEM},
            {"role": "user", "content": user_content},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "MonitoringBrief",
                "strict": True,
                "schema": MonitoringBrief.model_json_schema(),
            },
        },
    )
    return MonitoringBrief.model_validate_json(resp.choices[0].message.content)


# 4. Updated run() with the elif urgency == "watch" branch inserted
def run_with_watch(batch: SignalBatch) -> ResponseEngineResult:
    """Extended pipeline that handles the watch path with a monitoring brief."""
    classification = _classify(batch)

    if classification.urgency == "ignore":
        return ResponseEngineResult(
            brand_name=batch.brand_name,
            urgency=classification.urgency,
            classification=classification,
        )

    elif classification.urgency == "watch":
        monitoring_brief = _build_monitoring_brief(batch.brand_name, classification)
        print("\n--- Monitoring Brief ---")
        print(f"Summary:  {monitoring_brief.competitor_summary}")
        print("Keywords: " + ", ".join(monitoring_brief.tracking_keywords))
        return ResponseEngineResult(
            brand_name=batch.brand_name,
            urgency=classification.urgency,
            classification=classification,
            # counter_brief and content_pack remain None — no campaign generated
        )

    # urgency == "respond" — full fan-out as before
    brief = _build_brief(batch.brand_name, classification)

    tasks = {
        "email": lambda: _write_email(brief),
        "social": lambda: _write_social(brief),
        "blog": lambda: _write_blog(brief),
    }

    results: dict = {}
    with ThreadPoolExecutor(max_workers=3) as executor:
        futures = {executor.submit(fn): key for key, fn in tasks.items()}
        for future in as_completed(futures):
            results[futures[future]] = future.result()

    content_pack = ContentPack(
        topic=brief.topic,
        email=results["email"],
        social=results["social"],
        blog=results["blog"],
    )

    return ResponseEngineResult(
        brand_name=batch.brand_name,
        urgency=classification.urgency,
        classification=classification,
        counter_brief=brief,
        content_pack=content_pack,
    )


# Test the watch path with a mid-tier signal
SCENARIO_WATCH = SignalBatch(
    brand_name="Flowdesk",
    signals=[
        CompetitorSignal(
            source="press_release",
            headline="GrowthMail announces beta of AI-powered subject line tool",
            summary=(
                "GrowthMail, a mid-size email marketing competitor, has opened a beta for an "
                "AI subject line optimizer targeting SMB marketers. The tool is invite-only with "
                "a waitlist of 300 users. No pricing announced yet and the product is not "
                "feature-complete, but the AI angle is generating early buzz in marketing forums."
            ),
            competitor_name="GrowthMail",
            brand_name="Flowdesk",
        ),
    ],
)

print("Running watch scenario (expect: watch → monitoring brief only)")
result_watch = run_with_watch(SCENARIO_WATCH)
_print_result("SCENARIO WATCH — Emerging AI Feature", result_watch)